## Example 1: The sync and async clients expose scrape, crawl, map and search

In [ ]:
# Example 1: The sync and async clients expose scrape, crawl, map and search
import inspect
import firecrawl
from firecrawl import FirecrawlApp, AsyncFirecrawlApp

print("firecrawl", firecrawl.__version__)
app = FirecrawlApp(api_key="fc-test")  # constructing a client makes no request
assert app.api_key == "fc-test", app.api_key
for method in ("scrape", "crawl", "map", "search", "start_crawl", "get_crawl_status", "batch_scrape"):
    assert callable(getattr(app, method, None)), f"FirecrawlApp has no callable {method}"

async_app = AsyncFirecrawlApp(api_key="fc-test")
for method in ("scrape", "crawl", "map", "search"):
    assert inspect.iscoroutinefunction(getattr(async_app, method, None)), f"AsyncFirecrawlApp.{method} is not async"
print("OK -- FirecrawlApp and AsyncFirecrawlApp build offline and expose scrape, crawl, map and search")

## Example 2: Every API error type derives from FirecrawlError and carries the HTTP status

In [ ]:
# Example 2: Every API error type derives from FirecrawlError and carries the HTTP status
import firecrawl
from firecrawl import FirecrawlError

error_types = {name: getattr(firecrawl, name) for name in (
    "BadRequestError", "UnauthorizedError", "PaymentRequiredError", "WebsiteNotSupportedError",
    "RequestTimeoutError", "RateLimitError", "InternalServerError")}
for name, error_type in error_types.items():
    assert issubclass(error_type, FirecrawlError), f"{name} does not derive from FirecrawlError"

error = error_types["UnauthorizedError"]("Unauthorized: Invalid token", 401)
assert isinstance(error, FirecrawlError) and error.status_code == 401, error.status_code
assert str(error) == "Unauthorized: Invalid token", str(error)
print(f"OK -- {len(error_types)} error types derive from FirecrawlError:", sorted(error_types))

## Example 3: Scrape and map against a local stand-in for the Firecrawl API

In [ ]:
# Example 3: Scrape and map against a local stand-in for the Firecrawl API
import asyncio, http.server, json, threading
from firecrawl import FirecrawlApp, AsyncFirecrawlApp


class StubAPI(http.server.BaseHTTPRequestHandler):
    """Answers each POST with the reply set for its path, and records what the client sent."""
    replies = {}  # path -> (HTTP status, JSON body)
    seen = []     # (path, Authorization header, JSON body) of every request

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
        StubAPI.seen.append((self.path, self.headers.get("Authorization"), body))
        status, reply = StubAPI.replies[self.path]
        data = json.dumps(reply).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *args):  # keep the request log out of the notebook output
        pass


server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), StubAPI)
threading.Thread(target=server.serve_forever, daemon=True).start()
api_url = f"http://127.0.0.1:{server.server_address[1]}"

StubAPI.replies = {
    "/v2/scrape": (200, {"success": True, "data": {
        "markdown": "# Hello", "metadata": {"title": "Hello", "sourceURL": "https://example.com", "statusCode": 200}}}),
    "/v2/map": (200, {"success": True, "links": [{"url": "https://example.com/a", "title": "A"}, "https://example.com/b"]}),
}
app = FirecrawlApp(api_key="fc-test", api_url=api_url)

doc = app.scrape("https://example.com", formats=["markdown"])
assert doc.markdown == "# Hello" and doc.metadata.title == "Hello", doc
assert doc.metadata.source_url == "https://example.com" and doc.metadata.status_code == 200, doc.metadata
path, authorization, body = StubAPI.seen[-1]
assert path == "/v2/scrape" and authorization == "Bearer fc-test", (path, authorization)
assert body["url"] == "https://example.com" and body["formats"] == ["markdown"], body

site = app.map("https://example.com", limit=10)
assert [(link.url, link.title) for link in site.links] == [("https://example.com/a", "A"), ("https://example.com/b", None)]
assert StubAPI.seen[-1][0] == "/v2/map" and StubAPI.seen[-1][2]["limit"] == 10, StubAPI.seen[-1]


async def scrape_async():
    return await AsyncFirecrawlApp(api_key="fc-async", api_url=api_url).scrape("https://example.com")

async_doc = await scrape_async()
assert async_doc.markdown == "# Hello" and StubAPI.seen[-1][1] == "Bearer fc-async", StubAPI.seen[-1]
print(f"OK -- scrape (sync and async) and map sent {len(StubAPI.seen)} requests and parsed the replies")

## Example 4: API errors and the API key, checked against the local stand-in from Example 3

In [ ]:
# Example 4: API errors and the API key, checked against the local stand-in from Example 3
import os
from firecrawl import FirecrawlApp, FirecrawlError, UnauthorizedError, BadRequestError

app = FirecrawlApp(api_key="fc-test", api_url=api_url)
for status, error_type in ((401, UnauthorizedError), (400, BadRequestError)):
    StubAPI.replies["/v2/scrape"] = (status, {"success": False, "error": f"refused with {status}"})
    try:
        app.scrape("https://example.com")
        raise RuntimeError(f"a {status} reply did not raise")
    except FirecrawlError as error:
        assert type(error) is error_type and error.status_code == status, (type(error), error.status_code)
        assert f"refused with {status}" in str(error), str(error)
        print(f"{status} -> {type(error).__name__}")

# The key comes from the argument or from FIRECRAWL_API_KEY; with neither, no Authorization header is sent.
StubAPI.replies["/v2/scrape"] = (200, {"success": True, "data": {"markdown": "ok"}})
saved = os.environ.pop("FIRECRAWL_API_KEY", None)
try:
    FirecrawlApp(api_url=api_url).scrape("https://example.com")
    assert StubAPI.seen[-1][1] is None, StubAPI.seen[-1][1]
    os.environ["FIRECRAWL_API_KEY"] = "fc-from-env"
    FirecrawlApp(api_url=api_url).scrape("https://example.com")
    assert StubAPI.seen[-1][1] == "Bearer fc-from-env", StubAPI.seen[-1][1]
finally:
    os.environ.pop("FIRECRAWL_API_KEY", None)
    if saved is not None:
        os.environ["FIRECRAWL_API_KEY"] = saved
server.shutdown()
print("OK -- error replies raise the SDK's error types; the key is read from the argument or FIRECRAWL_API_KEY")

## Example 5: An invalid API key is refused by the real API (live call)

In [ ]:
# Example 5: An invalid API key is refused by the real API (live call)
from firecrawl import FirecrawlApp, FirecrawlError, UnauthorizedError

app = FirecrawlApp(api_key="fc-invalid-key-00000000", timeout=60)
try:
    app.scrape("https://example.com", formats=["markdown"])
    raise RuntimeError("the API accepted an invalid key")
except FirecrawlError as error:
    print(f"raised: {type(error).__name__} ({error.status_code}): {str(error)[:200]}")
    assert isinstance(error, UnauthorizedError) and error.status_code == 401, (type(error), error.status_code)
print("OK -- the API refuses an invalid key with UnauthorizedError")